# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [9]:
# =====================================================================
# 1. LOAD THE DATA
# =====================================================================
# We load the data from your 'data' subfolder 
# We include encoding='ISO-8859-1' to safely handle international text/characters
df_aviation = pd.read_csv('data/AviationData.csv', encoding='ISO-8859-1', low_memory=False)


# =====================================================================
# 2. INSPECT DATATYPES AND NANS
# =====================================================================
print("--- DATA SUMMARY & DATATYPES ---")
# .info() shows you every column, its data type, and how many non-blank rows it has
print(df_aviation.info())

print("\n--- MISSING VALUES (NaNs) COUNT ---")
# .isnull().sum() counts the exact number of missing/empty cells in every column
print(df_aviation.isnull().sum())


# =====================================================================
# 3. INSPECT SUMMARY STATISTICS
# =====================================================================
print("\n--- SUMMARY STATISTICS FOR NUMERICAL COLUMNS ---")
# .describe() automatically calculates mean, min, max, and medians for numeric columns
print(df_aviation.describe())

print("\n--- SUMMARY STATISTICS FOR TEXT/CATEGORICAL COLUMNS ---")
# Adding include=[object] lets you see top repeated values and unique counts for text columns
print(df_aviation.describe(include=[object]))

--- DATA SUMMARY & DATATYPES ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make 

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [10]:


# --- 1. INSPECT COLUMNS RELEVANT TO FILTERING ---
print("\n=== STEP 1: INSPECTING FILTER-CRITICAL COLUMNS ===")

# Convert Event.Date to datetime to extract the calendar year
df_aviation['Event.Date'] = pd.to_datetime(df_aviation['Event.Date'], errors='coerce')
df_aviation['Year'] = df_aviation['Event.Date'].dt.year

# Check missing values for our target filter variables
filter_cols = ['Year', 'Amateur.Built', 'Aircraft.Category', 'Make']
print("Missing values in columns used for filtering:")
print(df_aviation[filter_cols].isnull().sum())

print("\nValue counts for Amateur.Built (Raw):")
print(df_aviation['Amateur.Built'].value_counts(dropna=False))


# --- 2. REASONABLE IMPUTATIONS & STANDARDIZATIONS ---
print("\n=== STEP 2: APPLYING REASONABLE IMPUTATIONS & CLEANING ===")

# Create a copy to perform cleaning without affecting the original dataframe
df_working = df_aviation.copy()

# Imputation A: 'Amateur.Built'
df_working['Amateur.Built'] = df_working['Amateur.Built'].fillna('No')

# Standardization B: 'Make' (Manufacturer Name)
# Text data contains mixed casings and extra trailing spaces (e.g., 'Cessna', 'CESSNA ', 'CESSNA CO').
# Standardizing to clean uppercase ensures robust filtering and grouping later.
df_working['Make'] = df_working['Make'].str.upper().str.strip()


# --- 3. FILTER THE DATASET ---
print("\n=== STEP 3: FILTERING THE DATASET ===")

# Constraint 1: Operational Window (1983 onward for active aircraft up to a 40-year maximum lifetime)
df_filtered = df_working[df_working['Year'] >= 1983]

# Constraint 2: Build Integrity (Exclude custom home-built / experimental aircraft)
df_filtered = df_filtered[df_filtered['Amateur.Built'] == 'No']

# Constraint 3: Target Airplanes Only (Smart Filtering)
# Critical Implication: Prior to 2008, the NTSB rarely populated the 'Aircraft.Category' column.
# Standard filtering would delete 25 years of data. To fix this, we preserve rows where 
# Category is explicitly 'Airplane' OR where Category is missing (NaN) and the manufacturer 
# is NOT a known helicopter or balloon brand.
non_airplane_brands = [
    'BELL', 'ROBINSON', 'HUGHES', 'SCHWEIZER', 'SIKORSKY', 
    'EUROCOPTER', 'ENSTROM', 'BALLOON WORKS', 'AEROSPATIALE'
]

is_explicit_airplane = df_filtered['Aircraft.Category'] == 'Airplane'
is_likely_airplane = df_filtered['Aircraft.Category'].isna() & (~df_filtered['Make'].isin(non_airplane_brands))

# Combine the conditions to generate the final filtered airplane dataset
df_final_airplanes = df_filtered[is_explicit_airplane | is_likely_airplane].copy()


# --- FINAL VERIFICATION ---
print("\n=== CLEANING & FILTERING RESULTS ===")
print(f"Original Row Count: {len(df_aviation)}")
print(f"Final Filtered Airplane Row Count: {len(df_final_airplanes)}")
print(f"Active Timeline Window: {df_final_airplanes['Year'].min()} to {df_final_airplanes['Year'].max()}")


=== STEP 1: INSPECTING FILTER-CRITICAL COLUMNS ===
Missing values in columns used for filtering:
Year                     0
Amateur.Built          102
Aircraft.Category    56602
Make                    63
dtype: int64

Value counts for Amateur.Built (Raw):
Amateur.Built
No     80312
Yes     8475
NaN      102
Name: count, dtype: int64

=== STEP 2: APPLYING REASONABLE IMPUTATIONS & CLEANING ===

=== STEP 3: FILTERING THE DATASET ===

=== CLEANING & FILTERING RESULTS ===
Original Row Count: 88889
Final Filtered Airplane Row Count: 68574
Active Timeline Window: 1983 to 2022


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [11]:
#--- 1. IMPUTE MISSING INJURY VALUES ---
# Assumption: NaNs in injury logs denote that no individuals fell into that category.
# Imputing with 0 ensures numeric consistency for downstream math operations.
injury_columns = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']

for col in injury_columns:
    df_final_airplanes[col] = df_final_airplanes[col].fillna(0)


# --- 2. DERIVE TOTAL OCCUPANTS & SEVERE INJURY RATE ---
# Derived Column A: Total Occupants (Passengers + Crew)
df_final_airplanes['Total.Occupants'] = (
    df_final_airplanes['Total.Fatal.Injuries'] +
    df_final_airplanes['Total.Serious.Injuries'] +
    df_final_airplanes['Total.Minor.Injuries'] +
    df_final_airplanes['Total.Uninjured']
)

# Derived Column B: Serious/Fatal Injury Rate
# Measures the fraction of people on board who suffered serious or fatal injuries.
df_final_airplanes['Serious.Fatal.Injury.Rate'] = (
    df_final_airplanes['Total.Fatal.Injuries'] + df_final_airplanes['Total.Serious.Injuries']
) / df_final_airplanes['Total.Occupants']

# Handling Division by Zero: If Total.Occupants was 0, filling the resulting NaN with 0.0
df_final_airplanes['Serious.Fatal.Injury.Rate'] = df_final_airplanes['Serious.Fatal.Injury.Rate'].fillna(0.0)


print("\n=== STEP 5: CLEANING AIRCRAFT DAMAGE & CREATING DESTRUCTION METRIC ===")

# --- 3. CLEAN & BINARIZE AIRCRAFT DAMAGE ---
# Standardize text formatting to handle capitalization mismatches
df_final_airplanes['Aircraft.damage'] = df_final_airplanes['Aircraft.damage'].str.strip().str.capitalize()

# Derived Column C: Is.Destroyed (1 = Total Hull Loss, 0 = Hull Survived)
# This simplifies the categorical damage descriptors into a straightforward binary flag.
df_final_airplanes['Is.Destroyed'] = np.where(df_final_airplanes['Aircraft.damage'] == 'Destroyed', 1, 0)


# =====================================================================
# VERIFICATION AND SUMMARY STATISTICS
# =====================================================================
print("\n=== DATA TRANSFORMATION METRICS VERIFIED ===")
print(df_final_airplanes[['Total.Occupants', 'Serious.Fatal.Injury.Rate', 'Is.Destroyed']].describe())

print("\nSummary of Hull Loss vs. Survival counts:")
print(df_final_airplanes['Is.Destroyed'].value_counts())



=== STEP 5: CLEANING AIRCRAFT DAMAGE & CREATING DESTRUCTION METRIC ===

=== DATA TRANSFORMATION METRICS VERIFIED ===
       Total.Occupants  Serious.Fatal.Injury.Rate  Is.Destroyed
count     68574.000000               68574.000000  68574.000000
mean          7.208126                   0.266652      0.201913
std          31.249046                   0.426330      0.401431
min           0.000000                   0.000000      0.000000
25%           1.000000                   0.000000      0.000000
50%           2.000000                   0.000000      0.000000
75%           3.000000                   0.500000      0.000000
max         699.000000                   1.000000      1.000000

Summary of Hull Loss vs. Survival counts:
Is.Destroyed
0    54728
1    13846
Name: count, dtype: int64


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [12]:
print("\n=== CLEANING AIRCRAFT DAMAGE & CREATING DESTRUCTION METRIC ===")

# 1. Clean and Standardize the text data
# We remove any trailing spaces and force standard capitalization ('Destroyed', 'Substantial', etc.)
df_final_airplanes['Aircraft.damage'] = df_final_airplanes['Aircraft.damage'].str.strip().str.capitalize()

# 2. Construct the binary derived column: 'Is.Destroyed'
# np.where works like an IF statement: 
# IF Aircraft.damage is exactly 'Destroyed', give it a 1. OTHERWISE, give it a 0.
df_final_airplanes['Is.Destroyed'] = np.where(df_final_airplanes['Aircraft.damage'] == 'Destroyed', 1, 0)


# =====================================================================
# VERIFICATION AND SUMMARY STATISTICS
# =====================================================================
print("\n--- Standardized Damage Categories (Value Counts) ---")
print(df_final_airplanes['Aircraft.damage'].value_counts(dropna=False))

print("\n--- New Binary 'Is.Destroyed' Metric Summary ---")
print(df_final_airplanes['Is.Destroyed'].value_counts())

# Calculate the baseline destruction rate across the entire dataset
overall_destruction_rate = df_final_airplanes['Is.Destroyed'].mean() * 100
print(f"\nOverall Airplane Destruction Rate: {overall_destruction_rate:.2f}%")


=== CLEANING AIRCRAFT DAMAGE & CREATING DESTRUCTION METRIC ===

--- Standardized Damage Categories (Value Counts) ---
Aircraft.damage
Substantial    49439
Destroyed      13846
NaN             2789
Minor           2403
Unknown           97
Name: count, dtype: int64

--- New Binary 'Is.Destroyed' Metric Summary ---
Is.Destroyed
0    54728
1    13846
Name: count, dtype: int64

Overall Airplane Destruction Rate: 20.19%


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [14]:
print("\n=== STEP 6: STANDARDIZING AND FILTERING THE 'MAKE' COLUMN ===")

# 1. Define a robust function to clean corporate suffixes and consolidate names
def standardize_manufacturer(make_value):
    if pd.isna(make_value):
        return "UNKNOWN"
    
    # Force uppercase and remove surrounding spaces
    make_str = str(make_value).upper().strip()
    
    # Use keyword matching to consolidate variations into unified parent brands
    if 'CESSNA' in make_str:
        return 'CESSNA'
    if 'PIPER' in make_str:
        return 'PIPER'
    if 'BEECH' in make_str:
        return 'BEECH'
    if 'BOEING' in make_str:
        return 'BOEING'
    if 'MOONEY' in make_str:
        return 'MOONEY'
    if 'GRUMMAN' in make_str:
        return 'GRUMMAN'
    if 'AIR TRACTOR' in make_str:
        return 'AIR TRACTOR'
    if 'MCDONNELL DOUGLAS' in make_str or 'MCDONNELL-DOUGLAS' in make_str:
        return 'MCDONNELL DOUGLAS'
    if 'MAULE' in make_str:
        return 'MAULE'
    if 'CHAMPION' in make_str:
        return 'CHAMPION'
    if 'BELLANCA' in make_str:
        return 'BELLANCA'
    if 'AERONCA' in make_str:
        return 'AERONCA'
    
    return make_str

# Apply the standardization function to create a new clean column
df_final_airplanes['Make_Clean'] = df_final_airplanes['Make'].apply(standardize_manufacturer)


# 2. Apply a frequency threshold to keep only established manufacturers (Count >= 50)
# This removes single-instance typos and extremely rare aircraft lines.
make_frequency = df_final_airplanes['Make_Clean'].value_counts()

# Identify which makes meet our client criteria (>= 50 incidents)
valid_makes_threshold = make_frequency[make_frequency >= 50].index

# Filter the dataset to keep only rows containing these established brands
df_analysis_ready = df_final_airplanes[df_final_airplanes['Make_Clean'].isin(valid_makes_threshold)].copy()


# =====================================================================
# VERIFICATION AND SUMMARY STATISTICS
# =====================================================================
print("\n=== 'MAKE' STANDARDIZATION & FILTER COMPLETE ===")
print(f"Unique manufacturer names BEFORE mapping: {df_final_airplanes['Make'].nunique()}")
print(f"Unique manufacturer names AFTER mapping & thresholding: {df_analysis_ready['Make_Clean'].nunique()}")
print(f"Total rows preserved for exploratory analysis: {len(df_analysis_ready)}")

print("\n--- Top 15 Manufacturers Represented in Final Analysis Set ---")
print(df_analysis_ready['Make_Clean'].value_counts().head(15))


=== STEP 6: STANDARDIZING AND FILTERING THE 'MAKE' COLUMN ===

=== 'MAKE' STANDARDIZATION & FILTER COMPLETE ===
Unique manufacturer names BEFORE mapping: 1548
Unique manufacturer names AFTER mapping & thresholding: 70
Total rows preserved for exploratory analysis: 63495

--- Top 15 Manufacturers Represented in Final Analysis Set ---
Make_Clean
CESSNA               25823
PIPER                14181
BEECH                 5182
BOEING                2763
GRUMMAN               1524
MOONEY                1321
BELLANCA               980
AIR TRACTOR            900
AERONCA                610
CHAMPION               607
MCDONNELL DOUGLAS      590
MAULE                  573
STINSON                421
AERO COMMANDER         411
DE HAVILLAND           405
Name: count, dtype: int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [16]:
df = pd.read_csv('data/AviationData.csv', encoding='ISO-8859-1', low_memory=False)
# Convert dates and extract Year directly into the filter
df['Event.Date'] = pd.to_datetime(df['Event.Date'], errors='coerce')
df_filtered = df[(df['Event.Date'].dt.year >= 1983) & (df['Amateur.Built'].fillna('No') == 'No')].copy()

# Standardize text columns up-front to save repetitive typing later
df_filtered['Make'] = df_filtered['Make'].str.upper().str.strip()
df_filtered['Model_Clean'] = df_filtered['Model'].str.upper().str.strip()

# Smart Airplane Filter: Include 'Airplane' or NaNs, but exclude known helicopters/balloons
non_airplane_brands = ['BELL', 'ROBINSON', 'HUGHES', 'SCHWEIZER', 'SIKORSKY', 'EUROCOPTER', 'ENSTROM', 'BALLOON WORKS', 'AEROSPATIALE']
is_airplane = (df_filtered['Aircraft.Category'] == 'Airplane') | (df_filtered['Aircraft.Category'].isna() & ~df_filtered['Make'].isin(non_airplane_brands))
df_airplanes = df_filtered[is_airplane].copy()


# =====================================================================
# 2. FEATURE ENGINEERING (Injuries, Destruction, and Model Overlaps)
# =====================================================================

# Bulk fill all injury column NaNs with 0 in one line, then sum them up
injury_cols = ['Total.Fatal.Injuries', 'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured']
df_airplanes[injury_cols] = df_airplanes[injury_cols].fillna(0)
df_airplanes['Total.Occupants'] = df_airplanes[injury_cols].sum(axis=1)

# Calculate rate and natively protect against division by zero using .fillna(0)
df_airplanes['Serious.Fatal.Injury.Rate'] = ((df_airplanes['Total.Fatal.Injuries'] + df_airplanes['Total.Serious.Injuries']) / df_airplanes['Total.Occupants']).fillna(0)

# Track asset destruction (1 if 'Destroyed', 0 otherwise)
df_airplanes['Is.Destroyed'] = np.where(df_airplanes['Aircraft.damage'].str.strip().str.capitalize() == 'Destroyed', 1, 0)


# =====================================================================
# 3. MANUFACTURER CLEANING & FINAL ANALYSIS DATASET
# =====================================================================

# Clean top manufacturer brand names efficiently using vectorized string matching
df_airplanes['Make_Clean'] = df_airplanes['Make'] # Fallback default
for brand in ['CESSNA', 'PIPER', 'BEECH', 'BOEING', 'MOONEY', 'GRUMMAN', 'AIR TRACTOR', 'MAULE', 'CHAMPION', 'BELLANCA', 'AERONCA']:
    df_airplanes.loc[df_airplanes['Make'].str.contains(brand, na=False), 'Make_Clean'] = brand
df_airplanes.loc[df_airplanes['Make'].str.contains('MCDONNELL', na=False), 'Make_Clean'] = 'MCDONNELL DOUGLAS'

# Drop missing models and create the composite unique identifier
df_airplanes = df_airplanes.dropna(subset=['Model_Clean'])
df_airplanes['Unique_Plane_Model'] = df_airplanes['Make_Clean'] + " " + df_airplanes['Model_Clean']

# Filter out low-frequency makes (keeps only manufacturers with 50+ incidents)
make_counts = df_airplanes['Make_Clean'].value_counts()
df_analysis_ready = df_airplanes[df_airplanes['Make_Clean'].isin(make_counts[make_counts >= 50].index)].copy()


# =====================================================================
# PRINT QUICK OUTPUT VERIFICATION
# =====================================================================
print(f"Final Data Shape: {df_analysis_ready.shape}")
print(f"Unique Aircraft Configurations: {df_analysis_ready['Unique_Plane_Model'].nunique()}")

Final Data Shape: (63473, 37)
Unique Aircraft Configurations: 5461


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# List of text columns to clean simultaneously
text_operational_cols = ['Engine.Type', 'Weather.Condition', 'Purpose.of.flight', 'Broad.phase.of.flight']


for col in text_operational_cols:
    df_analysis_ready[col] = df_analysis_ready[col].astype(str).str.upper().str.strip()
    
    # Standardize 'UNKNOWN' variations within text columns (e.g., 'UNK' -> 'UNKNOWN')
    df_analysis_ready[col] = df_analysis_ready[col].replace(['UNK', 'NAN'], 'UNKNOWN')

# 2. Clean 'Number.of.Engines' column
# Ensure values are strictly numeric where possible, leaving structural NaNs as a clean category
df_analysis_ready['Number.of.Engines'] = pd.to_numeric(df_analysis_ready['Number.of.Engines'], errors='coerce')


# =====================================================================
# VERIFICATION AND CATEGORICAL SUMMARY PEAKS
# =====================================================================
print("\n=== OPERATIONAL COLUMNS STANDARDIZED SUCCESSFULLY ===")

print("\n--- Cleaned Weather Conditions ---")
print(df_analysis_ready['Weather.Condition'].value_counts(dropna=False).head(3))

print("\n--- Cleaned Engine Types (Top 5) ---")
print(df_analysis_ready['Engine.Type'].value_counts(dropna=False).head(5))

print("\n--- Cleaned Engine Counts ---")
print(df_analysis_ready['Number.of.Engines'].value_counts(dropna=False).head(5))

print("\n--- Cleaned Broad Phase of Flight (Top 5) ---")
print(df_analysis_ready['Broad.phase.of.flight'].value_counts(dropna=False).head(5))

print("\n--- Cleaned Purpose of Flight (Top 5) ---")
print(df_analysis_ready['Purpose.of.flight'].value_counts(dropna=False).head(5))


=== OPERATIONAL COLUMNS STANDARDIZED SUCCESSFULLY ===

--- Cleaned Weather Conditions ---
Weather.Condition
VMC        54401
IMC         4982
UNKNOWN     4090
Name: count, dtype: int64

--- Cleaned Engine Types (Top 5) ---
Engine.Type
RECIPROCATING    52237
UNKNOWN           5507
TURBO PROP        2749
TURBO FAN         2197
TURBO JET          534
Name: count, dtype: int64

--- Cleaned Engine Counts ---
Number.of.Engines
1.0    49134
2.0     9288
NaN     3885
3.0      430
4.0      400
Name: count, dtype: int64

--- Cleaned Broad Phase of Flight (Top 5) ---
Broad.phase.of.flight
UNKNOWN        18405
LANDING        12239
TAKEOFF         9165
CRUISE          7507
MANEUVERING     4863
Name: count, dtype: int64

--- Cleaned Purpose of Flight (Top 5) ---
Purpose.of.flight
PERSONAL              35551
UNKNOWN                9400
INSTRUCTIONAL          8182
AERIAL APPLICATION     3417
BUSINESS               3027
Name: count, dtype: int64


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [18]:
missing_percentages = df_analysis_ready.isnull().sum() / len(df_analysis_ready)

print("--- Missing Data Percentages per Column ---")
print(missing_percentages.sort_values(ascending=False).round(4) * 100)


# 2. Define the threshold constraint (Drop columns with > 50% missing data)
# .dropna(thresh=...) keeps columns that have at least 'X' non-NaN rows.
row_threshold = int(0.50 * len(df_analysis_ready))

# Execute the column drop
df_cleaned_final = df_analysis_ready.dropna(axis=1, thresh=row_threshold).copy()


# 3. Identify and report exactly which columns were dropped
dropped_columns = set(df_analysis_ready.columns) - set(df_cleaned_final.columns)


# =====================================================================
# VERIFICATION AND COMPACTION SUMMARY
# =====================================================================
print("\n=== COLUMN REMOVAL COMPLETE ===")
print(f"Columns removed from the dataset: {list(dropped_columns) if dropped_columns else 'None'}")
print(f"Original shape before column drop: {df_analysis_ready.shape}")
print(f"Final shape after column drop: {df_cleaned_final.shape}")
print("\n--- Remaining Clean Columns for Exploratory Analysis ---")
print(list(df_cleaned_final.columns))

--- Missing Data Percentages per Column ---
Schedule                     85.29
Air.carrier                  81.89
FAR.Description              70.32
Aircraft.Category            70.03
Longitude                    63.78
Latitude                     63.77
Airport.Code                 40.50
Airport.Name                 37.76
Publication.Date             17.55
Report.Status                 6.88
Number.of.Engines             6.12
Aircraft.damage               3.88
Registration.Number           1.47
Injury.Severity               1.32
Country                       0.17
Amateur.Built                 0.13
Location                      0.06
Broad.phase.of.flight         0.00
Event.Id                      0.00
Model_Clean                   0.00
Total.Occupants               0.00
Serious.Fatal.Injury.Rate     0.00
Total.Uninjured               0.00
Is.Destroyed                  0.00
Make_Clean                    0.00
Weather.Condition             0.00
Engine.Type                   0.00
Total.Minor

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [19]:
output_path = 'data/AviationData_Cleaned.csv'

# Export the DataFrame
# Crucial: index=False prevents Pandas from creating an extra, unnamed row-number column
df_cleaned_final.to_csv(output_path, index=False)